# Celery

Celery 适合稳定的后台任务系统，例如发送邮件、报表生成、视频转码、异步 Webhook 和批量同步。生产里通常拆成 worker 和 producer 两类进程。

- broker: Redis、RabbitMQ 等消息中间件。
- backend: 保存任务结果，常用 Redis 或数据库。
- worker: 执行任务。
- producer: 提交任务。

```zsh
celery -A python_notes.task_queues.celery.tasks worker --loglevel=info
python -m python_notes.task_queues.celery.client
```


In [ ]:
import os
from typing import Final

from celery import Celery

DEFAULT_REDIS_URL: Final[str] = 'redis://localhost:6379/0'
REDIS_URL: Final[str] = os.getenv('TASK_QUEUE_REDIS_URL', DEFAULT_REDIS_URL)

app = Celery('task_queue_examples', broker=REDIS_URL, backend=REDIS_URL)
app.conf.update(
    task_serializer='json',
    result_serializer='json',
    accept_content=['json'],
    timezone='Asia/Shanghai',
    enable_utc=False,
    task_track_started=True,
    task_time_limit=30,
    result_expires=3600,
)


@app.task(name='notebook.celery.add')
def celery_add(x: int, y: int) -> int:
    return x + y


@app.task(name='notebook.celery.multiply')
def celery_multiply(x: int, y: int) -> int:
    return x * y


## Canvas 工作流

Celery Canvas 可以组合任务：`chain()` 串行执行，`group()` 并行执行，`chord()` 在一组任务完成后执行汇总任务。


In [ ]:
from celery import chain, chord, group

from python_notes.task_queues.celery.tasks import add, multiply, sum_numbers

single = add.apply_async(args=(4, 6), countdown=1)
print('Task ID:', single.id)

chained = chain(add.s(2, 3), multiply.s(10)).apply_async()
grouped = group(add.s(index, index + 1) for index in range(3)).apply_async()
summarized = chord([multiply.s(index, 2) for index in range(1, 4)], sum_numbers.s()).apply_async()

print('Chain:', chained.get(timeout=10))
print('Group:', grouped.get(timeout=10))
print('Chord:', summarized.get(timeout=10))
